In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException, NoSuchElementException

# ----------------- 설정 및 초기화 -----------------
# 크롤링할 URL
URL = "https://www.greating.co.kr/market/marketDetail?itemId=166994&toggle=meals#modal__"
# 리뷰 데이터를 담을 리스트
all_reviews = []
# WebDriver 초기화 (Chrome이 설치되어 있고, 환경 변수에 등록되어 있다고 가정)
driver = webdriver.Chrome()

# ----------------- 함수 정의 -----------------

def scroll_to_element(driver, by, value):
    """지정된 요소까지 스크롤합니다."""
    # 상품 상세 정보 섹션 (리뷰 탭을 포함하는 상위 영역)
    try:
        element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((by, value))
        )
        # 해당 요소가 뷰포트 상단에 오도록 스크롤합니다.
        driver.execute_script("arguments[0].scrollIntoView({block: 'start'});", element)
        print(f"✅ 스크롤 완료: '{value}' 영역으로 이동.")
        time.sleep(1.5) # 스크롤 후 안정화 대기
    except TimeoutException:
        print(f"❌ '{value}' 요소를 찾을 수 없어 스크롤 실패.")

def click_review_tab(driver):
    """'상품리뷰' 탭을 클릭합니다."""
    # 캡처된 이미지에 보이는 '상품리뷰' 탭의 data-tab 속성: cont-review
    REVIEW_TAB_SELECTOR = "li[data-tab='cont-review']"
    
    try:
        # 탭이 클릭 가능할 때까지 대기
        review_tab = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
        )
        review_tab.click()
        print("✅ '상품리뷰' 탭 클릭 성공.")
        time.sleep(2) # 리뷰 내용 로딩 대기
    except (TimeoutException, ElementClickInterceptedException) as e:
        print(f"❌ '상품리뷰' 탭 클릭 실패: {e}. JavaScript로 강제 클릭 시도.")
        try:
            # 강제 클릭 시도
            driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
            time.sleep(2)
        except Exception as err:
            print(f"❌ JavaScript 강제 클릭도 실패: {err}")


def extract_reviews(driver):
    """현재 페이지에서 리뷰 정보를 추출합니다."""
    global all_reviews
    
    # 리뷰 리스트 아이템의 CSS Selector (실제 웹 구조에 맞게 조정)
    # #cont-review가 리뷰 내용이 담기는 div ID, 그 안의 ul > li 구조를 가정
    REVIEW_ITEMS_SELECTOR = "#cont-review > div.review-list > ul > li"
    
    try:
        # 리뷰 요소가 화면에 나타날 때까지 대기
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, REVIEW_ITEMS_SELECTOR))
        )
        review_elements = driver.find_elements(By.CSS_SELECTOR, REVIEW_ITEMS_SELECTOR)
    except TimeoutException:
        print("❌ 리뷰 목록 로딩 시간 초과 또는 요소가 없습니다.")
        return False
    
    if not review_elements:
        print("❌ 추출할 리뷰 요소가 없습니다. (페이지 구조 확인 필요)")
        return False
        
    for review_el in review_elements:
        data = {
            'User': 'N/A',
            'Rating': 'N/A',
            'Content': 'N/A',
            'Image': 'No Img'
        }
        
        try:
            # 작성자 (사용자 ID 클래스를 가정)
            data['User'] = review_el.find_element(By.CSS_SELECTOR, ".user-id").text
        except NoSuchElementException:
            pass
            
        try:
            # 평점 (별점 요소를 가정)
            # star-point 클래스 내부에 있는 span 요소의 style 속성 등으로 추출 가능하지만, 여기서는 텍스트로 대체
            # 예: data['Rating'] = review_el.find_element(By.CSS_SELECTOR, ".star-point").get_attribute("title")
            data['Rating'] = "평점 정보"
        except NoSuchElementException:
            pass
        
        try:
            # 리뷰 내용
            data['Content'] = review_el.find_element(By.CSS_SELECTOR, ".review-text").text
        except NoSuchElementException:
            pass
            
        try:
            # 이미지 여부 (Img 문자로 처리 요청)
            # 리뷰 아이템 내부에 이미지를 포함하는 요소(예: .review-img)를 찾으면 'Img'로 설정
            review_el.find_element(By.CSS_SELECTOR, ".review-img img")
            data['Image'] = "Img"
        except NoSuchElementException:
            pass
                
        all_reviews.append(data)
            
    print(f"✅ 현재 페이지에서 {len(review_elements)}개 리뷰 추출 완료.")
    print(f"⭐ 총 누적 리뷰 개수: {len(all_reviews)}개")
    return True

def go_to_next_page(driver, page_num):
    """다음 페이지 버튼을 찾아 클릭합니다."""
    # 페이지네이션 영역의 CSS Selector를 정확히 확인해야 합니다.
    # 일반적으로 페이지 번호를 담는 li 요소를 클릭합니다.
    try:
        # 다음 페이지 번호 (현재 페이지 번호 + 1)를 찾아 클릭
        next_page_link = driver.find_element(By.XPATH, f"//div[contains(@class, 'paging-cont')]//a[text()='{page_num + 1}']")
        next_page_link.click()
        print(f"➡️ {page_num + 1} 페이지로 이동합니다...")
        time.sleep(2) # 페이지 로딩 대기
        return True
    except NoSuchElementException:
        # 다음 페이지 번호가 없거나, '다음' 버튼 등을 사용해야 하는 경우 (XPath 변경 필요)
        try:
             # '>' 모양의 다음 버튼을 찾는 일반적인 XPath
            next_button = driver.find_element(By.XPATH, "//div[contains(@class, 'paging-cont')]//a[contains(@class, 'btn-next')]")
            if 'disabled' in next_button.get_attribute("class"):
                return False # 버튼이 비활성화된 경우
            next_button.click()
            print("➡️ '다음' 버튼 클릭 후 페이지 이동합니다...")
            time.sleep(2)
            return True
        except NoSuchElementException:
            return False # 다음 페이지 버튼이 없음

# ----------------- 메인 실행 로직 -----------------
try:
    print("🚀 크롤링 시작...")
    driver.get(URL)
    
    # 1. 스크롤: 리뷰 탭이 포함된 상위 섹션으로 이동
    SCROLL_TARGET_SELECTOR = "section.marketDetail_sect.detail"
    scroll_to_element(driver, By.CSS_SELECTOR, SCROLL_TARGET_SELECTOR)
    
    # 2. '상품리뷰' 탭 클릭
    click_review_tab(driver)
    
    # 3. 리뷰 크롤링 및 페이지네이션 루프
    page_num = 1
    # 100 페이지를 넘지 않도록 제한 (무한 루프 방지)
    MAX_PAGES = 100 
    
    while page_num <= MAX_PAGES:
        print(f"\n--- {page_num} 페이지 크롤링 시작 ---")
        
        # 리뷰 추출
        if not extract_reviews(driver):
            # 첫 페이지에 리뷰가 없거나 추출 실패 시 종료
            if page_num == 1:
                break 
                
        # 다음 페이지로 이동 시도
        if not go_to_next_page(driver, page_num):
            print("✅ 마지막 페이지에 도달했거나 다음 페이지 버튼이 없어 크롤링을 종료합니다.")
            break
            
        page_num += 1

except Exception as e:
    print(f"\n❌ 치명적인 오류 발생: {e}")

finally:
    driver.quit()
    print("\n✅ WebDriver 종료.")

    # ----------------- 데이터 저장 (Excel) -----------------
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        
        # 엑셀 파일로 저장 (주피터 노트북이 실행되는 디렉토리에 저장됩니다)
        EXCEL_FILENAME = "greating_product_reviews.xlsx"
        df.to_excel(EXCEL_FILENAME, index=False)
        print(f"\n🎉 크롤링 완료! 총 {len(all_reviews)}개의 리뷰를 '{EXCEL_FILENAME}'에 저장했습니다.")
    else:
        print("\n⚠️ 수집된 리뷰 데이터가 없습니다.")

🚀 크롤링 시작...
❌ 'section.marketDetail_sect.detail' 요소를 찾을 수 없어 스크롤 실패.
❌ '상품리뷰' 탭 클릭 실패: Message: element click intercepted: Element <li class="tab-menu__list tnb-area__list" data-tab="cont-review" id="reviewTab">...</li> is not clickable at point (492, 869). Other element would receive the click: <div class="btn-option">...</div>
  (Session info: chrome=141.0.7390.123); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
0   chromedriver                        0x0000000104257108 chromedriver + 6119688
1   chromedriver                        0x000000010424e80a chromedriver + 6084618
2   chromedriver                        0x0000000103cea1a6 chromedriver + 430502
3   chromedriver                        0x0000000103d43840 chromedriver + 796736
4   chromedriver                        0x0000000103d4168b chromedriver + 788107
5   chromedriver                        0x0000000103d3ed1

In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException, NoSuchElementException, WebDriverException

# ----------------- 설정 및 초기화 -----------------
URL = "https://www.greating.co.kr/market/marketDetail?itemId=166994&toggle=meals#modal__"
all_reviews = []
# Chrome WebDriver 초기화
driver = webdriver.Chrome()

# ----------------- 함수 정의 -----------------

def scroll_to_bottom(driver):
    """페이지 최하단까지 스크롤하여 모든 요소를 로드합니다."""
    # JavaScript를 사용하여 스크롤을 페이지 끝까지 내립니다.
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    print("✅ 페이지 최하단까지 스크롤 완료.")
    time.sleep(1.5) # 로딩 대기

def click_review_tab(driver):
    """'상품리뷰' 탭을 클릭합니다. 클릭이 가로막힐 경우 JavaScript로 강제 클릭합니다."""
    REVIEW_TAB_SELECTOR = "li[data-tab='cont-review']"
    
    try:
        # 탭 요소가 DOM에 존재할 때까지 대기
        review_tab = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
        )
        
        # 1. 일반 클릭 시도
        review_tab.click()
        print("✅ '상품리뷰' 탭 일반 클릭 성공.")
        time.sleep(2) # 리뷰 내용 로딩 대기
        return
        
    except (TimeoutException, ElementClickInterceptedException) as e:
        # 2. 클릭이 가로막히거나 시간 초과 발생 시, JavaScript로 강제 클릭
        try:
            print(f"⚠️ 클릭 가로막힘 발생: {type(e).__name__}. JavaScript로 강제 클릭 시도.")
            driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
            print("✅ '상품리뷰' 탭 JavaScript 강제 클릭 성공.")
            time.sleep(2)
        except Exception as err:
            print(f"❌ JavaScript 강제 클릭도 실패: {err}")
            raise # 최종 실패 시 예외 발생

def extract_reviews(driver):
    """현재 페이지에서 리뷰 정보를 추출합니다."""
    global all_reviews
    REVIEW_ITEMS_SELECTOR = "#cont-review > div.review-list > ul > li"
    
    # ... (리뷰 추출 로직은 이전과 동일하므로, 여기서는 생략하고 핵심 로직만 유지) ...
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, REVIEW_ITEMS_SELECTOR))
        )
        review_elements = driver.find_elements(By.CSS_SELECTOR, REVIEW_ITEMS_SELECTOR)
    except TimeoutException:
        print("❌ 리뷰 목록 로딩 시간 초과 또는 요소가 없습니다.")
        return False
        
    if not review_elements:
        print("❌ 추출할 리뷰 요소가 없습니다.")
        return False
        
    for review_el in review_elements:
        data = {'User': 'N/A', 'Rating': 'N/A', 'Content': 'N/A', 'Image': 'No Img'}
        
        try:
            # 작성자 (클래스명을 .user-id로 가정)
            data['User'] = review_el.find_element(By.CSS_SELECTOR, ".user-id").text
        except NoSuchElementException: pass
            
        try:
            # 리뷰 내용
            data['Content'] = review_el.find_element(By.CSS_SELECTOR, ".review-text").text
        except NoSuchElementException: pass
            
        try:
            # 이미지 여부 (Img 문자로 처리)
            review_el.find_element(By.CSS_SELECTOR, ".review-img img")
            data['Image'] = "Img"
        except NoSuchElementException: pass
                
        all_reviews.append(data)
            
    print(f"✅ 현재 페이지에서 {len(review_elements)}개 리뷰 추출 완료. (총 누적: {len(all_reviews)}개)")
    return True

def go_to_next_page(driver, page_num):
    """다음 페이지 버튼을 찾아 클릭합니다."""
    
    # 1. 숫자 링크 클릭 시도
    try:
        next_page_link = driver.find_element(By.XPATH, f"//div[contains(@class, 'paging-cont')]//a[text()='{page_num + 1}']")
        next_page_link.click()
        print(f"➡️ {page_num + 1} 페이지(숫자 링크)로 이동합니다...")
        time.sleep(2) 
        return True
    except NoSuchElementException:
        # 2. '다음' 버튼 클릭 시도
        try:
            next_button = driver.find_element(By.XPATH, "//div[contains(@class, 'paging-cont')]//a[contains(@class, 'btn-next')]")
            if 'disabled' in next_button.get_attribute("class"):
                return False 
            next_button.click()
            print("➡️ '다음' 버튼 클릭 후 페이지 이동합니다...")
            time.sleep(2)
            return True
        except NoSuchElementException:
            return False 

# ----------------- 메인 실행 로직 -----------------
try:
    print("🚀 크롤링 시작...")
    driver.get(URL)
    
    # 1. 스크롤: 페이지 하단으로 스크롤하여 리뷰 탭이 화면에 나타나도록 함
    scroll_to_bottom(driver)
    
    # 2. '상품리뷰' 탭 클릭 (강제 클릭 로직 포함)
    click_review_tab(driver)
    
    # 3. 리뷰 크롤링 및 페이지네이션 루프
    page_num = 1
    MAX_PAGES = 100 
    
    while page_num <= MAX_PAGES:
        print(f"\n--- {page_num} 페이지 크롤링 시작 ---")
        
        # 리뷰 추출
        if not extract_reviews(driver) and page_num == 1:
            break # 첫 페이지에 리뷰가 없으면 종료
                
        # 다음 페이지로 이동 시도
        if not go_to_next_page(driver, page_num):
            print("✅ 마지막 페이지에 도달했거나 다음 페이지 버튼이 없어 크롤링을 종료합니다.")
            break
            
        page_num += 1

except WebDriverException as e:
    print(f"\n❌ WebDriver 관련 오류 발생 (브라우저 또는 드라이버 문제): {e}")
except Exception as e:
    print(f"\n❌ 일반 오류 발생: {e}")

finally:
    driver.quit()
    print("\n✅ WebDriver 종료.")

    # ----------------- 데이터 저장 (Excel) -----------------
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        EXCEL_FILENAME = "greating_product_reviews.xlsx"
        df.to_excel(EXCEL_FILENAME, index=False)
        print(f"\n🎉 크롤링 완료! 총 {len(all_reviews)}개의 리뷰를 '{EXCEL_FILENAME}'에 저장했습니다.")
    else:
        print("\n⚠️ 수집된 리뷰 데이터가 없습니다.")

🚀 크롤링 시작...
✅ 페이지 최하단까지 스크롤 완료.
✅ '상품리뷰' 탭 일반 클릭 성공.

--- 1 페이지 크롤링 시작 ---
❌ 리뷰 목록 로딩 시간 초과 또는 요소가 없습니다.

✅ WebDriver 종료.

⚠️ 수집된 리뷰 데이터가 없습니다.


In [4]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException, NoSuchElementException, WebDriverException

# ----------------- 설정 및 초기화 -----------------
URL = "https://www.greating.co.kr/market/marketDetail?itemId=166994&toggle=meals#modal__"
all_reviews = []
# Chrome WebDriver 초기화
driver = webdriver.Chrome()

# ----------------- 함수 정의 -----------------

def scroll_to_bottom(driver):
    """페이지 최하단까지 스크롤하여 모든 요소를 로드하고, 리뷰 탭이 뷰포트 안에 들어오도록 합니다."""
    # JavaScript를 사용하여 스크롤을 페이지 끝까지 내립니다.
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    print("✅ 페이지 최하단까지 스크롤 완료.")
    time.sleep(2) # 로딩 대기

def click_review_tab(driver):
    """'상품리뷰' 탭을 클릭합니다. 클릭이 가로막힐 경우 JavaScript로 강제 클릭합니다."""
    # 캡처된 이미지에 보이는 '상품리뷰' 탭의 Selector
    REVIEW_TAB_SELECTOR = "li[data-tab='cont-review']"
    
    try:
        # 탭 요소가 DOM에 존재하고 클릭 가능할 때까지 대기
        review_tab = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
        )
        
        # 1. 일반 클릭 시도
        review_tab.click()
        print("✅ '상품리뷰' 탭 일반 클릭 성공.")
        time.sleep(3) # ★★★ 리뷰 내용 로딩을 위한 대기 시간을 충분히 확보 (3초)
        return
        
    except (TimeoutException, ElementClickInterceptedException) as e:
        # 2. 클릭이 가로막히거나 시간 초과 발생 시, JavaScript로 강제 클릭
        try:
            print(f"⚠️ 클릭 오류 발생: {type(e).__name__}. JavaScript로 강제 클릭 시도.")
            review_tab = driver.find_element(By.CSS_SELECTOR, REVIEW_TAB_SELECTOR)
            driver.execute_script("arguments[0].click();", review_tab)
            print("✅ '상품리뷰' 탭 JavaScript 강제 클릭 성공.")
            time.sleep(3) # ★★★ 리뷰 내용 로딩을 위한 대기 시간을 충분히 확보
        except Exception as err:
            print(f"❌ JavaScript 강제 클릭도 실패: {err}")
            raise # 최종 실패 시 예외 발생

def extract_reviews(driver):
    """현재 페이지에서 리뷰 정보를 추출합니다."""
    global all_reviews
    
    # ★★★ 리뷰 목록 Selector: #cont-review 내부에서 가장 일반적인 리스트 아이템을 지정 (li)
    # 만약 실행 후에도 실패하면, 이 Selector를 F12 개발자 도구로 정확하게 확인하여 수정해야 합니다.
    REVIEW_ITEMS_SELECTOR = "#cont-review li"  
    
    try:
        # 리뷰 요소가 화면에 나타날 때까지 대기 시간을 15초로 늘림
        WebDriverWait(driver, 15).until( 
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, REVIEW_ITEMS_SELECTOR))
        )
        review_elements = driver.find_elements(By.CSS_SELECTOR, REVIEW_ITEMS_SELECTOR)
    except TimeoutException:
        print("❌ 리뷰 목록 로딩 시간 초과 또는 요소가 없습니다. (Selector 확인 필수)")
        return False
    
    if not review_elements:
        print("❌ 추출할 리뷰 요소가 없습니다.")
        return False
        
    for review_el in review_elements:
        data = {
            'User': 'N/A', 
            'Rating': 'N/A', 
            'Content': 'N/A', 
            'Image': 'No Img'
        }
        
        # -- 데이터 추출 (클래스명 가정) --
        try:
            data['User'] = review_el.find_element(By.CSS_SELECTOR, ".user-id").text
        except NoSuchElementException: pass
            
        try:
            # 평점 정보는 실제 클래스명에 따라 추출해야 함. (일단 문자열로 처리)
            data['Rating'] = "평점 정보" 
        except NoSuchElementException: pass
        
        try:
            data['Content'] = review_el.find_element(By.CSS_SELECTOR, ".review-text").text
        except NoSuchElementException: pass
            
        try:
            # 이미지 여부 확인 (요청대로 'Img' 문자열로 처리)
            review_el.find_element(By.CSS_SELECTOR, "img") 
            data['Image'] = "Img"
        except NoSuchElementException: pass
                
        all_reviews.append(data)
            
    print(f"✅ 현재 페이지에서 {len(review_elements)}개 리뷰 추출 완료. (총 누적: {len(all_reviews)}개)")
    return True

def go_to_next_page(driver, page_num):
    """다음 페이지 버튼을 찾아 클릭합니다."""
    
    # 1. 숫자 링크 클릭 시도
    try:
        # 다음 페이지 번호에 해당하는 <a> 태그를 찾아 클릭
        next_page_link = driver.find_element(By.XPATH, f"//div[contains(@class, 'paging-cont')]//a[text()='{page_num + 1}']")
        next_page_link.click()
        print(f"➡️ {page_num + 1} 페이지(숫자 링크)로 이동합니다...")
        time.sleep(2.5) # 페이지 로딩 대기
        return True
    except NoSuchElementException:
        # 2. '다음' 버튼 클릭 시도
        try:
            next_button = driver.find_element(By.XPATH, "//div[contains(@class, 'paging-cont')]//a[contains(@class, 'btn-next')]")
            if 'disabled' in next_button.get_attribute("class"):
                return False 
            next_button.click()
            print("➡️ '다음' 버튼 클릭 후 페이지 이동합니다...")
            time.sleep(2.5)
            return True
        except NoSuchElementException:
            return False 

# ----------------- 메인 실행 로직 -----------------
try:
    print("🚀 크롤링 시작...")
    driver.get(URL)
    
    # 1. 스크롤: 페이지 하단으로 스크롤하여 리뷰 탭이 화면에 나타나도록 함
    scroll_to_bottom(driver)
    
    # 2. '상품리뷰' 탭 클릭 (강제 클릭 로직 포함)
    click_review_tab(driver)
    
    # 3. 리뷰 크롤링 및 페이지네이션 루프
    page_num = 1
    MAX_PAGES = 100 
    
    while page_num <= MAX_PAGES:
        print(f"\n--- {page_num} 페이지 크롤링 시작 ---")
        
        # 리뷰 추출
        success = extract_reviews(driver)
        if not success and page_num == 1:
            # 첫 페이지에 리뷰가 없으면 Selector 문제로 간주하고 종료
            break 
                
        # 다음 페이지로 이동 시도
        if not go_to_next_page(driver, page_num):
            print("✅ 마지막 페이지에 도달했거나 다음 페이지 버튼이 없어 크롤링을 종료합니다.")
            break
            
        page_num += 1

except WebDriverException as e:
    print(f"\n❌ WebDriver 관련 오류 발생 (브라우저 또는 드라이버 문제): {e}")
except Exception as e:
    print(f"\n❌ 일반 오류 발생: {e}")

finally:
    driver.quit()
    print("\n✅ WebDriver 종료.")

    # ----------------- 데이터 저장 (Excel) -----------------
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        EXCEL_FILENAME = "greating_product_reviews.xlsx"
        df.to_excel(EXCEL_FILENAME, index=False)
        print(f"\n🎉 크롤링 완료! 총 {len(all_reviews)}개의 리뷰를 '{EXCEL_FILENAME}'에 저장했습니다.")
    else:
        print("\n⚠️ 수집된 리뷰 데이터가 없습니다.")

🚀 크롤링 시작...
✅ 페이지 최하단까지 스크롤 완료.
✅ '상품리뷰' 탭 일반 클릭 성공.

--- 1 페이지 크롤링 시작 ---
❌ 리뷰 목록 로딩 시간 초과 또는 요소가 없습니다. (Selector 확인 필수)

✅ WebDriver 종료.

⚠️ 수집된 리뷰 데이터가 없습니다.


In [22]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

# ----------------- 설정 및 초기화 -----------------
URL = "https://www.greating.co.kr/market/marketDetail?itemId=166994&toggle=meals#modal__"
all_reviews = []
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# ----------------- 함수 정의 -----------------

def scroll_to_view(driver, selector, align_to_top=False):
    """특정 요소가 화면에 보이도록 스크롤합니다."""
    try:
        element = driver.find_element(By.CSS_SELECTOR, selector)
        driver.execute_script(f"arguments[0].scrollIntoView({str(align_to_top).lower()});", element)
        if align_to_top:
            print(f"✅ '{selector}' 요소가 뷰포트 상단에 보이도록 스크롤 완료.")
        else:
             print(f"✅ '{selector}' 요소가 뷰포트 중앙에 보이도록 스크롤 완료.")
        time.sleep(1) 
    except NoSuchElementException:
        print(f"❌ '{selector}' 요소를 찾을 수 없어 스크롤 실패.")
        pass

def scroll_to_bottom(driver):
    """페이지 최하단까지 스크롤합니다."""
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    print("✅ 페이지 최하단까지 스크롤 완료.")
    time.sleep(2) 

def click_review_tab(driver):
    """'상품리뷰' 탭을 클릭하고 리뷰 섹션을 화면에 고정합니다."""
    REVIEW_TAB_SELECTOR = "li#reviewTab"
    
    try:
        review_tab = WebDriverWait(driver, 15).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
        )
        
        # ★★★ 오류 해결 핵심: JavaScript 강제 클릭 사용 ★★★
        driver.execute_script("arguments[0].click();", review_tab)
        print("✅ '상품리뷰' 탭 JavaScript 강제 클릭 성공.")
        
        # 탭 클릭 후 로딩 대기 시간
        time.sleep(12) 
        
        # 스크롤 조정: #cont-review를 뷰포트 중앙에 배치하여 리뷰 로드 유도
        scroll_to_view(driver, "#cont-review", align_to_top=False)
        
        return
        
    except Exception as e:
        # 오류 메시지를 명확하게 출력
        print(f"❌ '상품리뷰' 탭 클릭 또는 스크롤 중 오류 발생: {e}")
        raise 

def extract_reviews(driver):
    """현재 페이지에서 리뷰 정보를 추출합니다."""
    global all_reviews
    
    # Selector 정의
    REVIEW_SECTION_SELECTOR = "#cont-review"
    CONTENT_SELECTOR = "p.body-14-regular"
    USER_ID_SELECTOR = ".user-id"
    DATE_SELECTOR = "span.body-12-regular" 
    REVIEW_CANDIDATE_SELECTOR = f"{REVIEW_SECTION_SELECTOR} div" 
    
    try:
        # 대기 조건: 작성자 ID가 DOM에 나타날 때까지 기다립니다. (45초로 최대 확보)
        WebDriverWait(driver, 45).until( 
            EC.presence_of_element_located((By.CSS_SELECTOR, f"{REVIEW_SECTION_SELECTOR} {USER_ID_SELECTOR}"))
        )
        
        # 모든 잠재적 리뷰 컨테이너 요소를 가져옵니다.
        all_candidate_elements = driver.find_elements(By.CSS_SELECTOR, REVIEW_CANDIDATE_SELECTOR)
        
        review_elements = []
        
        # 필터링: 유효한 리뷰 아이템(작성자 ID와 리뷰 내용을 모두 포함)만 추출
        for el in all_candidate_elements:
            try:
                el.find_element(By.CSS_SELECTOR, USER_ID_SELECTOR)
                el.find_element(By.CSS_SELECTOR, CONTENT_SELECTOR)
                review_elements.append(el)
            except NoSuchElementException:
                continue
        
        review_elements = list(set(review_elements))

    except TimeoutException:
        print("❌ 리뷰 목록 로딩 시간 초과 (45초 초과). (작성자 ID가 DOM에 로드되지 않음)")
        return False
    except Exception as e:
        print(f"❌ 리뷰 아이템 탐색 중 알 수 없는 오류: {e}")
        return False
    
    if not review_elements:
        print("❌ 추출할 리뷰 요소가 없습니다. (작성자 ID와 리뷰 내용이 함께 포함된 DIV를 찾지 못함)")
        return False
        
    for review_el in review_elements:
        data = {
            'User': 'N/A', 
            'Date': 'N/A',
            'Purpose': 'N/A', 
            'Meal_Amount': 'N/A',
            'Option': 'N/A',
            'Content': 'N/A', 
            'Image': 'No Img'
        }
        
        # --- 데이터 추출 (이전 시도의 성공적인 XPath/CSS 로직 재사용) ---
        
        # 1. 한줄평(내용) 추출
        try:
            data['Content'] = review_el.find_element(By.CSS_SELECTOR, CONTENT_SELECTOR).text
        except NoSuchElementException: pass
        
        # 2. 작성자 ID 추출
        try:
            data['User'] = review_el.find_element(By.CSS_SELECTOR, USER_ID_SELECTOR).text
        except NoSuchElementException: pass

        # 3. 날짜 추출
        try:
            date_el = review_el.find_element(By.XPATH, f".//{USER_ID_SELECTOR}/following-sibling::span[contains(@class, '{DATE_SELECTOR.split('.')[-1]}')]")
            data['Date'] = date_el.text.strip()
        except NoSuchElementException: pass

        # 4. 주문 목적 / 5. 평소 식사량 추출
        try:
            purpose_label = review_el.find_element(By.XPATH, ".//span[text()='주문 목적']/following-sibling::span[1]")
            data['Purpose'] = purpose_label.text.strip()
            
            meal_label = review_el.find_element(By.XPATH, ".//span[text()='평소 식사량']/following-sibling::span[1]")
            data['Meal_Amount'] = meal_label.text.strip()
        except NoSuchElementException: pass
        
        # 6. 구매 옵션 추출
        try:
            option_el = review_el.find_element(By.XPATH, ".//p[contains(text(), '구매옵션')]/following-sibling::p[1]")
            data['Option'] = option_el.text.strip()
        except NoSuchElementException: pass

        # 7. 이미지 여부
        try:
            review_el.find_element(By.CSS_SELECTOR, "img") 
            data['Image'] = "Img"
        except NoSuchElementException: pass
                
        all_reviews.append(data)
            
    print(f"✅ 현재 페이지에서 {len(review_elements)}개 리뷰 추출 완료. (총 누적: {len(all_reviews)}개)")
    return True

def go_to_next_page(driver, page_num):
    """다음 페이지 버튼을 찾아 클릭합니다. 숫자 버튼(button 태그)을 XPath로 찾습니다."""
    
    NEXT_PAGE_XPATH = f"//button[normalize-space(text())='{page_num + 1}']"
    
    try:
        next_page_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, NEXT_PAGE_XPATH))
        )
        
        driver.execute_script("arguments[0].click();", next_page_button)
        print(f"➡️ {page_num + 1} 페이지 버튼 클릭으로 이동합니다...")
        time.sleep(3) 
        
        scroll_to_view(driver, "#cont-review", align_to_top=False)
        
        return True
    except TimeoutException:
         print(f"❌ {page_num + 1} 페이지 버튼이 시간 내에 활성화되지 않아 다음 페이지로 이동 실패. 마지막 페이지이거나 페이지네이션 오류.")
         return False
    except NoSuchElementException:
        return False 
    except Exception as e:
        print(f"⚠️ 다음 페이지 클릭 오류 발생: {e}. 다음 페이지로 이동 실패.")
        return False


# ----------------- 메인 실행 로직 -----------------
try:
    print("🚀 크롤링 시작...")
    driver.get(URL)
    
    # 1. 스크롤: 페이지 하단으로 스크롤 (탭이 보이도록)
    scroll_to_bottom(driver)
    
    # 2. '상품리뷰' 탭 클릭 및 스크롤 (리뷰 목록 고정)
    click_review_tab(driver)
    
    # 3. 리뷰 크롤링 및 페이지네이션 루프 (1페이지만 테스트)
    page_num = 1
    MAX_PAGES = 1 
    
    while page_num <= MAX_PAGES:
        print(f"\n--- {page_num} 페이지 크롤링 시작 ---")
        
        success = extract_reviews(driver)
        if not success:
            print("❗ 1페이지 리뷰 추출 실패. 크롤링을 중단합니다.")
            break 
                
        page_num += 1

except Exception as e:
    print(f"\n❌ 치명적인 오류 발생: {e}")

finally:
    driver.quit()
    print("\n✅ WebDriver 종료.")

    # ----------------- 데이터 저장 (Excel) -----------------
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        EXCEL_FILENAME = "greating_product_reviews_page1.xlsx"
        df.to_excel(EXCEL_FILENAME, index=False)
        print(f"\n🎉 크롤링 완료! 총 {len(all_reviews)}개의 리뷰를 '{EXCEL_FILENAME}'에 저장했습니다.")
    else:
        print("\n⚠️ 수집된 리뷰 데이터가 없습니다.")

🚀 크롤링 시작...
✅ 페이지 최하단까지 스크롤 완료.
✅ '상품리뷰' 탭 JavaScript 강제 클릭 성공.
✅ '#cont-review' 요소가 뷰포트 중앙에 보이도록 스크롤 완료.

--- 1 페이지 크롤링 시작 ---
❌ 리뷰 목록 로딩 시간 초과 (45초 초과). (작성자 ID가 DOM에 로드되지 않음)
❗ 1페이지 리뷰 추출 실패. 크롤링을 중단합니다.

✅ WebDriver 종료.

⚠️ 수집된 리뷰 데이터가 없습니다.


In [23]:
pip install undetected-chromedriver selenium pandas

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'undetected-chromedriver' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'undetected-chromedriver'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47048 sha256=a24b11e54d7c71ac5177d23ed737d8ddd396c7bd1239d146fa694f5768402b89
  Stored in directory: /Users/mac/Library/Caches/pip/wheels/7a/5f/c1/06f68421cc7172ef51504631252870bcb3a2fdf3b6a025f362
Successfully built undetected-chromedriver
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [undetected-chromedriver]cted-chromedriver]
Note: you may need to restart 

In [26]:
import time
import pandas as pd
# 웹사이트 자동화 감지를 우회하기 위해 undetected_chromedriver 사용
import undetected_chromedriver as uc 
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

# ----------------- 설정 및 초기화 -----------------
URL = "https://www.greating.co.kr/market/marketDetail?itemId=166994&toggle=meals#modal__"
all_reviews = []

# undetected_chromedriver를 사용하여 WebDriver 초기화
driver = uc.Chrome()

# ----------------- 함수 정의 -----------------

def scroll_to_view(driver, selector, align_to_top=False):
    """특정 요소가 화면에 보이도록 스크롤합니다."""
    try:
        element = driver.find_element(By.CSS_SELECTOR, selector)
        driver.execute_script(f"arguments[0].scrollIntoView({str(align_to_top).lower()});", element)
        if align_to_top:
            print(f"✅ '{selector}' 요소가 뷰포트 상단에 보이도록 스크롤 완료.")
        else:
             print(f"✅ '{selector}' 요소가 뷰포트 중앙에 보이도록 스크롤 완료.")
        time.sleep(1) 
    except NoSuchElementException:
        print(f"❌ '{selector}' 요소를 찾을 수 없어 스크롤 실패.")
        pass

def scroll_to_bottom(driver):
    """페이지 최하단까지 스크롤합니다."""
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    print("✅ 페이지 최하단까지 스크롤 완료.")
    time.sleep(2) 

def click_review_tab(driver):
    """'상품리뷰' 탭을 클릭하고 리뷰 섹션을 화면에 고정합니다."""
    REVIEW_TAB_SELECTOR = "li#reviewTab"
    
    try:
        review_tab = WebDriverWait(driver, 15).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, REVIEW_TAB_SELECTOR))
        )
        
        # 'Element Click Intercepted' 오류를 해결하기 위해 JavaScript 강제 클릭 사용
        driver.execute_script("arguments[0].click();", review_tab)
        print("✅ '상품리뷰' 탭 JavaScript 강제 클릭 성공. (자동화 방지 우회)")
        
        # 탭 클릭 후 리뷰 데이터 로딩을 위한 넉넉한 대기 시간 확보
        time.sleep(12) 
        
        # 스크롤 조정: #cont-review를 뷰포트 중앙에 배치
        scroll_to_view(driver, "#cont-review", align_to_top=False)
        
        return
        
    except Exception as e:
        print(f"❌ '상품리뷰' 탭 클릭 또는 스크롤 중 오류 발생: {e}")
        raise 

def extract_reviews(driver):
    """현재 페이지에서 리뷰 정보를 추출합니다."""
    global all_reviews
    
    # Selector 정의
    REVIEW_SECTION_SELECTOR = "#cont-review"
    CONTENT_SELECTOR = "p.body-14-regular" # 리뷰 내용 텍스트 (최종 대기 기준)
    USER_ID_SELECTOR = ".user-id"
    DATE_SELECTOR = "span.body-12-regular" 
    REVIEW_CANDIDATE_SELECTOR = f"{REVIEW_SECTION_SELECTOR} div" # 모든 잠재적 리뷰 컨테이너 DIV
    
    try:
        # ★★★ 최종 대기 조건: 리뷰 내용 텍스트가 'DOM에 존재'할 때까지 기다립니다. (Visibility 대신 Presence 사용)
        WebDriverWait(driver, 30).until( 
            EC.presence_of_element_located((By.CSS_SELECTOR, f"{REVIEW_SECTION_SELECTOR} {CONTENT_SELECTOR}"))
        )
        print("✅ 리뷰 내용 P 태그가 DOM에 존재하는 것을 확인했습니다. 추출을 시작합니다.")
        
        # 모든 잠재적 리뷰 컨테이너 요소를 가져옵니다.
        all_candidate_elements = driver.find_elements(By.CSS_SELECTOR, REVIEW_CANDIDATE_SELECTOR)
        
        review_elements = []
        
        # 필터링: 유효한 리뷰 아이템(작성자 ID와 리뷰 내용을 모두 포함)만 추출
        for el in all_candidate_elements:
            try:
                # 작성자 ID와 리뷰 내용이 모두 포함된 DIV만 유효한 리뷰 아이템으로 간주
                el.find_element(By.CSS_SELECTOR, USER_ID_SELECTOR)
                el.find_element(By.CSS_SELECTOR, CONTENT_SELECTOR)
                review_elements.append(el)
            except NoSuchElementException:
                continue
        
        # 중복 제거
        review_elements = list(set(review_elements))

    except TimeoutException:
        print("❌ 리뷰 목록 로딩 시간 초과 (30초 초과). (리뷰 내용 P 태그가 DOM에 로드되지 않음)")
        return False
    except Exception as e:
        print(f"❌ 리뷰 아이템 탐색 중 알 수 없는 오류: {e}")
        return False
    
    if not review_elements:
        print("❌ 추출할 리뷰 요소가 없습니다. (리뷰 텍스트는 로드되었으나, 추출용 컨테이너 DIV를 찾지 못함)")
        return False
        
    for review_el in review_elements:
        data = {
            'User': 'N/A', 
            'Date': 'N/A',
            'Purpose': 'N/A', 
            'Meal_Amount': 'N/A',
            'Option': 'N/A',
            'Content': 'N/A', 
            'Image': 'No Img'
        }
        
        # --- 데이터 추출 로직 (XPath/CSS 조합) ---
        
        # 1. 한줄평(내용) 추출
        try:
            data['Content'] = review_el.find_element(By.CSS_SELECTOR, CONTENT_SELECTOR).text
        except NoSuchElementException: pass
        
        # 2. 작성자 ID 추출
        try:
            data['User'] = review_el.find_element(By.CSS_SELECTOR, USER_ID_SELECTOR).text
        except NoSuchElementException: pass

        # 3. 날짜 추출 (XPath: .user-id의 다음 형제 span 중 body-12-regular 클래스를 가진 요소)
        try:
            date_el = review_el.find_element(By.XPATH, f".//{USER_ID_SELECTOR}/following-sibling::span[contains(@class, '{DATE_SELECTOR.split('.')[-1]}')]")
            data['Date'] = date_el.text.strip()
        except NoSuchElementException: pass

        # 4. 주문 목적 / 5. 평소 식사량 추출 (XPath: 텍스트 레이블의 다음 형제 span)
        try:
            purpose_label = review_el.find_element(By.XPATH, ".//span[text()='주문 목적']/following-sibling::span[1]")
            data['Purpose'] = purpose_label.text.strip()
            
            meal_label = review_el.find_element(By.XPATH, ".//span[text()='평소 식사량']/following-sibling::span[1]")
            data['Meal_Amount'] = meal_label.text.strip()
        except NoSuchElementException: pass
        
        # 6. 구매 옵션 추출 (XPath: '구매옵션' P 태그의 다음 형제 P 태그)
        try:
            option_el = review_el.find_element(By.XPATH, ".//p[contains(text(), '구매옵션')]/following-sibling::p[1]")
            data['Option'] = option_el.text.strip()
        except NoSuchElementException: pass

        # 7. 이미지 여부
        try:
            review_el.find_element(By.CSS_SELECTOR, "img") 
            data['Image'] = "Img"
        except NoSuchElementException: pass
                
        all_reviews.append(data)
            
    print(f"✅ 현재 페이지에서 {len(review_elements)}개 리뷰 추출 완료. (총 누적: {len(all_reviews)}개)")
    return True

def go_to_next_page(driver, page_num):
    """다음 페이지 버튼을 찾아 클릭합니다. (1페이지만 크롤링하므로 사용 안 함)"""
    return False 


# ----------------- 메인 실행 로직 -----------------
try:
    print("🚀 크롤링 시작...")
    driver.get(URL)
    
    # 1. 스크롤: 페이지 하단으로 스크롤
    scroll_to_bottom(driver)
    
    # 2. '상품리뷰' 탭 클릭 및 스크롤
    click_review_tab(driver)
    
    # 3. 리뷰 크롤링 (1페이지만)
    page_num = 1
    MAX_PAGES = 1 
    
    while page_num <= MAX_PAGES:
        print(f"\n--- {page_num} 페이지 크롤링 시작 ---")
        
        success = extract_reviews(driver)
        if not success:
            print("❗ 1페이지 리뷰 추출 실패. 크롤링을 중단합니다.")
            break 
                
        page_num += 1

except Exception as e:
    print(f"\n❌ 치명적인 오류 발생: {e}")

finally:
    driver.quit()
    print("\n✅ WebDriver 종료.")

    # ----------------- 데이터 저장 (Excel) -----------------
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        EXCEL_FILENAME = "greating_product_reviews_page1.xlsx"
        df.to_excel(EXCEL_FILENAME, index=False)
        print(f"\n🎉 크롤링 완료! 총 {len(all_reviews)}개의 리뷰를 '{EXCEL_FILENAME}'에 저장했습니다.")
    else:
        print("\n⚠️ 수집된 리뷰 데이터가 없습니다.")

🚀 크롤링 시작...
✅ 페이지 최하단까지 스크롤 완료.
✅ '상품리뷰' 탭 JavaScript 강제 클릭 성공. (자동화 방지 우회)
✅ '#cont-review' 요소가 뷰포트 중앙에 보이도록 스크롤 완료.

--- 1 페이지 크롤링 시작 ---
❌ 리뷰 목록 로딩 시간 초과 (30초 초과). (리뷰 내용 P 태그가 DOM에 로드되지 않음)
❗ 1페이지 리뷰 추출 실패. 크롤링을 중단합니다.

✅ WebDriver 종료.

⚠️ 수집된 리뷰 데이터가 없습니다.
